# Feature-Rich MF6 Workflow

This notebook builds a small but more realistic MODFLOW 6 model using `myflopy`.
It uses a synthetic Voronoi grid, but the workflow includes the same kinds of features used in larger real models:

- CHD
- DRN from polygons
- GHB from polygons
- recharge from polygons
- UZF derived from recharge
- LAK from a lake polygon
- SFR from a stream line
- region and group registration for later querying


In [ ]:
from pathlib import Path
import shutil

import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Polygon

import myflopy as mf
from myflopy.modflow.mf6.drn import DRNFromVector
from myflopy.modflow.mf6.ghb import GHBFromVector
from myflopy.modflow.mf6.lakes import LAKBuilder
from myflopy.modflow.mf6.recharge import RCHFromVector
from myflopy import SFRBuilder
from myflopy import ModelContext, UZFBuilder
from myflopy.modflow.mf6.simulation.discretization import DisvGrid, TemporalDiscretization
from myflopy.modflow.mf6.simulation.packages import (
    CHD,
    InitialConditions,
    KFlow,
    OutputControl,
    Recharge,
    Storage,
)


In [ ]:
workspace = Path.cwd().parents[1] / "artifacts" / "feature_rich_model_workflow"
if workspace.exists():
    shutil.rmtree(workspace, ignore_errors=True)
workspace.mkdir(parents=True, exist_ok=True)

workspace

In [ ]:
def write_gpkg(path: Path, gdf: gpd.GeoDataFrame) -> Path:
    gdf.to_file(path, driver="GPKG")
    return path


def build_four_cell_vor() -> mf.VoronoiGridPlus:
    verts = np.array(
        [
            [0.0, 0.0], [1.0, 0.0], [2.0, 0.0],
            [0.0, 1.0], [1.0, 1.0], [2.0, 1.0],
            [0.0, 2.0], [1.0, 2.0], [2.0, 2.0],
        ],
        dtype=float,
    )
    iverts = [[0, 3, 4, 1], [1, 4, 5, 2], [3, 6, 7, 4], [4, 7, 8, 5]]
    xcyc = np.array(
        [[0.5, 0.5], [1.5, 0.5], [0.5, 1.5], [1.5, 1.5]],
        dtype=float,
    )
    vor = mf.VoronoiGridPlus(verts=verts, iverts=iverts, xcyc=xcyc)
    vor.gdf_topbtm = gpd.GeoDataFrame(
        {
            "geometry": vor.gdf_vorPolys.geometry,
            0: [12.0, 12.0, 12.0, 12.0],
            1: [0.0, 0.0, 0.0, 0.0],
        },
        geometry="geometry",
        crs=vor.crs,
    )
    return vor


vor = build_four_cell_vor()
vor.gdf_vorPolys

In [ ]:
model = mf.SimulationBase(name="feat_rich_demo", mf_folder_path=workspace, vor=vor, nper=2)

DisvGrid(vor=vor, model=model, top=[12.0] * 4, bottom=[[0.0] * 4], nlay=1)
TemporalDiscretization(model=model, per_len=1, num_steps=1, multiplier=1.0)
InitialConditions(model=model, vor=vor, nlay=1, strt=[11.0] * 4)
KFlow(model=model, k=[1.0] * 4, k33_vert=[1.0] * 4, save_specific_discharge=False)
Storage(model=model, sto_steady={0: True}, sto_transient={1: True})
OutputControl(model=model)
CHD(model=model, stress_period_data={0: [[(0, 1), 10.5]], 1: [[(0, 1), 10.25]]})

model.model_output_folder_path

In [ ]:
drain_path = write_gpkg(
    workspace / "drain.gpkg",
    gpd.GeoDataFrame(
        {
            "name": ["northwest_drain"],
            "height": [1.0],
            "cond": [5.0],
            "layer": [1],
            "min_elev": [2.0],
        },
        geometry=[Polygon([(0.0, 1.0), (0.95, 1.0), (0.95, 2.0), (0.0, 2.0)])],
        crs=vor.crs,
    ),
)

ghb_path = write_gpkg(
    workspace / "ghb.gpkg",
    gpd.GeoDataFrame(
        {
            "name": ["northeast_ghb"],
            "elev": [11.25],
            "height": [0.0],
            "cond": [7.0],
            "layer": [1],
            "min_elev": [10.0],
        },
        geometry=[Polygon([(1.05, 1.0), (2.0, 1.0), (2.0, 2.0), (1.05, 2.0)])],
        crs=vor.crs,
    ),
)

recharge_path = write_gpkg(
    workspace / "recharge.gpkg",
    gpd.GeoDataFrame(
        {
            "zone": ["left_recharge", "right_recharge"],
            "rch_0": [0.015, 0.01],
            "rch_1": [0.02, 0.012],
        },
        geometry=[
            Polygon([(0.0, 0.0), (0.95, 0.0), (0.95, 1.0), (0.0, 1.0)]),
            Polygon([(1.05, 0.0), (2.0, 0.0), (2.0, 1.0), (1.05, 1.0)]),
        ],
        crs=vor.crs,
    ),
)

lake_path = write_gpkg(
    workspace / "lake.gpkg",
    gpd.GeoDataFrame(
        {"name": ["lake_0"]},
        geometry=[Polygon([(0.0, 0.0), (0.95, 0.0), (0.95, 0.95), (0.0, 0.95)])],
        crs=vor.crs,
    ),
)

stream_path = write_gpkg(
    workspace / "stream.gpkg",
    gpd.GeoDataFrame(
        {"name": ["stream_0"]},
        geometry=[LineString([(0.1, 1.5), (1.9, 1.5)])],
        crs=vor.crs,
    ),
)


In [ ]:
drn_builder = DRNFromVector(model=model, vor=vor, shp_gpkg=drain_path, uid="name", idomain=[1, 1, 1, 1])
drn_dict = drn_builder.from_vector(
    edges_only=True,
    register_regions=True,
    region_name_prefix="drn_group",
    combined_region_name="all_drains",
    region_tags=["drn"],
    overwrite_regions=True,
)
mf.modflow.mf6.simulation.packages.Drains(model=model, stress_period_data=drn_dict)

ghb_builder = GHBFromVector(model=model, vor=vor, shp_gpkg=ghb_path, uid="name", idomain=[1, 1, 1, 1])
ghb_dict = ghb_builder.from_vector(
    register_regions=True,
    region_name_prefix="ghb_group",
    combined_region_name="all_ghb",
    region_tags=["ghb"],
    overwrite_regions=True,
)
mf.modflow.mf6.simulation.packages.GHB(model=model, stress_period_data=ghb_dict)

recharge_builder = RCHFromVector(
    model=model,
    vor=vor,
    shp_gpkg=recharge_path,
    uid="zone",
    rch_fields=["rch_0", "rch_1"],
    rch_fields_to_pers=[0, 1],
    background_rch=0.0,
    grid_type="disv",
    limit_to_k33=False,
)
rch_dict = recharge_builder.from_vector(
    register_regions=True,
    region_name_prefix="rch_zone",
    combined_region_name="all_rch",
    region_tags=["rch"],
    overwrite_regions=True,
)
Recharge(model=model, vor=vor, rch_dict=rch_dict)

uzf_context = ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain))
uzf_cells = [(0, cell) for cell in range(vor.ncpl)]
uzf_finf = {period: [{tuple(row[0]): row[1] for row in rows}.get(cellid, 0.0) for cellid in uzf_cells] for period, rows in rch_dict.items()}
uzf = UZFBuilder(
    context=uzf_context,
    nper=model.nper,
    cells=uzf_cells,
    vks=1.0,
    thtr=0.1,
    thts=0.3,
    thti=0.2,
    finf=uzf_finf,
)
uzf.build().build(model.gwf)
model.add_region_from_cells("uzf_all", uzf.uzf_cells, category="boundary", package="uzf", tags=["uzf"], overwrite=True)

lak = LAKBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    lakes=[lake_path],
    lake_id_field="name",
    starting_stage=11.0,
    lake_bottom=9.0,
    bed_leakance=0.1,
    connection_modes="automatic",
    status="ACTIVE",
    mover=False,
)
lak.build().build(model.gwf)
for lake_id, cells in lak.lake_cells.items():
    model.add_region_from_cells(f"lake_zone_{lake_id}", [(0, cell) for cell in cells], category="boundary", package="lak", tags=["lak"], geometry=lak.lake_table.loc[lake_id].geometry, overwrite=True)
model.add_region_from_cells("all_lakes", [(0, cell) for cells in lak.lake_cells.values() for cell in cells], category="boundary", package="lak", tags=["lak"], overwrite=True)

sfr = SFRBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    streams=[stream_path],
    width=1.0,
    gradient=0.001,
    roughness=0.03,
    streambed_k=1.0,
    streambed_thickness=1.0,
)
sfr.build().build(model.gwf)
model.add_region_from_cells("all_streams", [(0, cell) for cells in sfr.stream_cells.values() for cell in cells], category="boundary", package="sfr", tags=["sfr"], overwrite=True)

model.add_group(
    "boundary_features",
    members=["all_drains", "all_ghb", "all_rch", "uzf_all", "all_lakes", "all_streams"],
)


In [ ]:
success, _ = model.run_simulation()
success

In [ ]:
model.list_regions()

In [ ]:
model.list_groups()

In [ ]:
resolved_cells, trace = model.resolve_region_cells_with_trace("boundary_features")
resolved_cells, trace

In [ ]:
model.region_heads("boundary_features", per=1)[["cell", "elev", "region"]]

In [ ]:
model.gwf.plot()